# Silicon dose map in a simple tokamak

This example estimates silicon absorbed dose (KERMA) on a mesh tally across a simplified tokamak geometry, **without adding silicon to the geometry**. This is useful for predicting dose in silicon-based detectors or electronics that might be placed inside or near the tokamak.

The simulation also computes biological effective dose for comparison, using ICRP-116 dose coefficients.

The key technique used here is `multiply_density=False` on the heating tally with `nuclides=["Si28"]`. This tells OpenMC to score microscopic cross sections for Si28 at a density of 1 atom/barn-cm, regardless of what material actually fills each cell.

First import packages and configure the nuclear data path.

In [ ]:
import math
from pathlib import Path

from matplotlib import colormaps
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import openmc

# Setting the cross section path to the correct location in the docker image.
# If you are running this outside the docker image you will have to change this path to your local cross section path.
openmc.config['cross_sections'] = Path.home() / 'nuclear_data' / 'cross_sections.xml'

## Materials

We define the tokamak materials. Note that **none of these contain silicon** (except the concrete slab) — that is the whole point of using `multiply_density=False` to score Si dose independently of the local material.

In [ ]:
iron = openmc.Material(name='iron')
iron.add_element('Fe', 1.0)
iron.set_density("g/cm3", 7.874)

lithium = openmc.Material(name='lithium')
lithium.add_element('Li', 1.0)
lithium.set_density("g/cm3", 0.534)

tungsten = openmc.Material(name='tungsten')
tungsten.add_element('W', 1.0)
tungsten.set_density("g/cm3", 19.3)

concrete = openmc.Material(name='concrete')
concrete.add_element('O', 0.532)
concrete.add_element('Si', 0.337)
concrete.add_element('Ca', 0.044)
concrete.add_element('Al', 0.034)
concrete.add_element('Fe', 0.014)
concrete.add_element('H', 0.023)
concrete.add_element('Na', 0.016)
concrete.set_density("g/cm3", 2.3)

## Geometry

A simplified tokamak built from concentric spherical shells and a central column cylinder:

- **Central column** (tungsten) — inner cylinder of radius 150 cm
- **Plasma region** (void) — inside the inner sphere, outside the central column
- **First wall** (iron) — 20 cm thick shell around the plasma
- **Blanket** (lithium) — 80 cm thick shell around the first wall
- **Outer vessel** (void) — region outside the blanket
- **Concrete slab** — 100 cm thick slab below the tokamak

In [ ]:
R0 = 500.0              # major radius [cm]
FW_THICK = 20.0         # first-wall thickness [cm]
BLK_THICK = 80.0        # blanket thickness [cm]
CC_RADIUS = 150.0       # center-column cylinder radius [cm]
CC_EXTEND = 10.0        # center column extends beyond blanket [cm]

inner_r = R0
fw_r = R0 + FW_THICK
outer_r = R0 + FW_THICK + BLK_THICK
half_height = outer_r + CC_EXTEND

# surfaces
inner_sphere = openmc.Sphere(r=inner_r)
fw_sphere = openmc.Sphere(r=fw_r)
outer_sphere = openmc.Sphere(r=outer_r)
center_cyl = openmc.ZCylinder(r=CC_RADIUS)
top_plane = openmc.ZPlane(z0=half_height, boundary_type='vacuum')
bot_plane = openmc.ZPlane(z0=-half_height)
slab_bot_plane = openmc.ZPlane(z0=-half_height - 100, boundary_type='vacuum')
outer_cyl = openmc.ZCylinder(r=outer_r, boundary_type='vacuum')

# regions
plasma_region = -inner_sphere & +center_cyl
fw_region = +inner_sphere & -fw_sphere & +center_cyl
blanket_region = +fw_sphere & -outer_sphere & +center_cyl
center_col_region = -center_cyl & -top_plane & +bot_plane
outer_region = +outer_sphere & +center_cyl & -top_plane & +bot_plane & -outer_cyl
slab_region = -bot_plane & +slab_bot_plane & -outer_cyl

# cells
plasma_cell = openmc.Cell(region=plasma_region, name='plasma')           # void
fw_cell = openmc.Cell(region=fw_region, name='first_wall', fill=iron)
blanket_cell = openmc.Cell(region=blanket_region, name='blanket', fill=lithium)
cc_cell = openmc.Cell(region=center_col_region, name='center_column', fill=tungsten)
outer_cell = openmc.Cell(region=outer_region, name='outer_vessel')       # void
slab_cell = openmc.Cell(region=slab_region, name='concrete_slab', fill=concrete)

geometry = openmc.Geometry([plasma_cell, fw_cell, blanket_cell, cc_cell, outer_cell, slab_cell])

## Source

A 14.1 MeV D-T ring source at the plasma midplane, positioned halfway between the central column and the first wall.

In [ ]:
source_r = CC_RADIUS + (R0 - CC_RADIUS) / 2.0   # halfway between CC and FW
source = openmc.IndependentSource(
    space=openmc.stats.CylindricalIndependent(
        r=openmc.stats.Discrete([source_r], [1.0]),
        phi=openmc.stats.Uniform(0.0, 2 * math.pi),
        z=openmc.stats.Discrete([0.0], [1.0]),
    ),
    energy=openmc.stats.Discrete([14.1e6], [1.0]),    # 14.1 MeV
    angle=openmc.stats.Isotropic(),
)

settings = openmc.Settings(
    run_mode='fixed source',
    photon_transport=True,
    particles=50_000,
    batches=10,
    source=source,
    seed=42,
)

## Tallies

We set up four mesh tallies:

**Biological dose** (neutron + photon):
- Scores `flux` weighted by ICRP-116 dose coefficients via `EnergyFunctionFilter`
- Result is in pSv per source particle per voxel

**Silicon absorbed dose / KERMA** (neutron + photon):
- Scores `heating` with `nuclides=["Si28"]` and `multiply_density=False`
- `multiply_density=False` sets atom density to 1 atom/barn-cm = 10^24 atoms/cm^3 internally
- Result is in eV/cm^3 per source particle, assuming that fictional density
- The actual Si density cancels out when converting to Gy — both interactions and target mass scale linearly with density

In [ ]:
mesh = openmc.RegularMesh.from_domain(geometry, dimension=500_000)
mesh_filter = openmc.MeshFilter(mesh)

print(f"Mesh: {mesh.dimension} = {math.prod(mesh.dimension)} voxels")
print(f"  lower_left  = {mesh.lower_left}")
print(f"  upper_right = {mesh.upper_right}")
print(f"  width       = {mesh.width}")

In [ ]:
# Biological dose tallies
neutron_filter = openmc.ParticleFilter("neutron")
photon_filter = openmc.ParticleFilter("photon")

energy_bins_n, dose_coeffs_n = openmc.data.dose_coefficients(
    particle="neutron", geometry="AP"
)
efilter_n = openmc.EnergyFunctionFilter(energy_bins_n, dose_coeffs_n)
efilter_n.interpolation = "cubic"

energy_bins_p, dose_coeffs_p = openmc.data.dose_coefficients(
    particle="photon", geometry="AP"
)
efilter_p = openmc.EnergyFunctionFilter(energy_bins_p, dose_coeffs_p)
efilter_p.interpolation = "cubic"

bio_n_tally = openmc.Tally(name="bio_dose_neutron")
bio_n_tally.filters = [mesh_filter, neutron_filter, efilter_n]
bio_n_tally.scores = ["flux"]

bio_p_tally = openmc.Tally(name="bio_dose_photon")
bio_p_tally.filters = [mesh_filter, photon_filter, efilter_p]
bio_p_tally.scores = ["flux"]

In [ ]:
# Si absorbed dose (KERMA) tallies
si_n_tally = openmc.Tally(name="si_dose_neutron")
si_n_tally.filters = [mesh_filter, neutron_filter]
si_n_tally.scores = ["heating"]
si_n_tally.nuclides = ["Si28"]
si_n_tally.multiply_density = False

si_p_tally = openmc.Tally(name="si_dose_photon")
si_p_tally.filters = [mesh_filter, photon_filter]
si_p_tally.scores = ["heating"]
si_p_tally.nuclides = ["Si28"]
si_p_tally.multiply_density = False

## Run the simulation

In [ ]:
model = openmc.Model(
    geometry=geometry,
    settings=settings,
    tallies=[bio_n_tally, bio_p_tally, si_n_tally, si_p_tally],
)

print(f"Running: {settings.particles} particles x {settings.batches} batches")
model.run(apply_tally_results=True)
print("Done.")

## Post-processing

Convert raw tally results to physical dose rates:

**Biological dose:** The `EnergyFunctionFilter` already folded in dose coefficients (pSv-cm^2), so the tally result is in pSv per source particle. Multiply by source rate and convert pSv to Sv.

**Silicon dose:** The heating score gives eV/cm^3 per source particle at 1 atom/barn-cm density. To convert to Gy/s:
1. Multiply by source rate (n/s)
2. Multiply by 1.602e-19 (J/eV)
3. Divide by the fictional density corresponding to 1 atom/barn-cm of Si28

In [ ]:
SOURCE_RATE = 1e20  # neutrons per second

# Si dose conversion constants
A_SI = 28.085   # g/mol (Si atomic mass)
N_A = 6.022e23  # mol^-1 (Avogadro)
eV_TO_J = 1.602e-19  # J/eV
rho_fictional = 1e24 * A_SI / N_A * 1e-3  # kg/cm^3

si_n = si_n_tally.mean * SOURCE_RATE * eV_TO_J / rho_fictional   # Gy/s
si_p = si_p_tally.mean * SOURCE_RATE * eV_TO_J / rho_fictional   # Gy/s
si_total = si_n + si_p                                            # Gy/s

bio_n = bio_n_tally.mean * SOURCE_RATE * 1e-12   # Sv/s
bio_p = bio_p_tally.mean * SOURCE_RATE * 1e-12   # Sv/s
bio_total = bio_n + bio_p                         # Sv/s

print(f"Biological dose (neutron) range: {bio_n.min():.3e} to {bio_n.max():.3e} Sv/s")
print(f"Biological dose (photon)  range: {bio_p.min():.3e} to {bio_p.max():.3e} Sv/s")
print(f"Biological dose (total)   range: {bio_total.min():.3e} to {bio_total.max():.3e} Sv/s")
print(f"Silicon dose    (neutron) range: {si_n.min():.3e} to {si_n.max():.3e} Gy/s")
print(f"Silicon dose    (photon)  range: {si_p.min():.3e} to {si_p.max():.3e} Gy/s")
print(f"Silicon dose    (total)   range: {si_total.min():.3e} to {si_total.max():.3e} Gy/s")

## Plot results

A 2x3 grid of XZ slices through the midplane (y=0), comparing biological dose (top row) and silicon dose (bottom row) for neutrons, photons, and their sum.

In [ ]:
nx, ny, nz = mesh.dimension
y_mid = ny // 2  # index closest to y=0

x_extent = [mesh.lower_left[0], mesh.upper_right[0]]
z_extent = [mesh.lower_left[2], mesh.upper_right[2]]
extent = x_extent + z_extent

def xz_slice(data):
    """Extract an XZ slice at y=0 from tally data ordered (nz, ny, nx)."""
    return data.reshape(nz, ny, nx)[:, y_mid, :]

cmap = colormaps.get_cmap('inferno').resampled(12)

plot_data = [
    [xz_slice(bio_n), xz_slice(bio_p), xz_slice(bio_total)],
    [xz_slice(si_n),  xz_slice(si_p),  xz_slice(si_total)],
]
titles = [
    ['Biological dose neutron (Sv/s)', 'Biological dose photon (Sv/s)', 'Biological dose total (Sv/s)'],
    ['Silicon dose neutron (Gy/s)',    'Silicon dose photon (Gy/s)',    'Silicon dose total (Gy/s)'],
]
units = ['Sv/s', 'Sv/s', 'Sv/s', 'Gy/s', 'Gy/s', 'Gy/s']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row in range(2):
    for col in range(3):
        ax = axes[row, col]
        im = ax.imshow(
            plot_data[row][col], origin='lower', extent=extent,
            aspect='equal', norm=mcolors.LogNorm(), cmap=cmap,
        )
        geometry.plot(basis='xz', outline='only', axes=ax, pixels=1_000_000)
        ax.set_title(titles[row][col])
        ax.set_xlabel('X (cm)')
        ax.set_ylabel('Z (cm)')
        fig.colorbar(im, ax=ax, label=units[row * 3 + col])

fig.tight_layout()
plt.savefig('si_vs_bio_dose_xz.png', dpi=150)
plt.show()
print("Saved si_vs_bio_dose_xz.png")

**Learning Outcomes:**

- Scoring dose for a nuclide (Si28) that is not present in the geometry using `multiply_density=False`.
- Understanding the unit conversions from OpenMC tally results to physical dose rates (Gy/s and Sv/s).
- Comparing silicon absorbed dose (KERMA) with biological effective dose across a tokamak geometry.
- Using mesh tallies with `EnergyFunctionFilter` for biological dose and `heating` scores for material-specific KERMA.